In [7]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os
import sys
import networkx as nx
import json
from typing import Dict, List

# In Jupyter notebooks, __file__ is not defined
# Use the current working directory instead
notebook_dir = os.path.abspath('')
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '../..')))
from concept_creator.neo4j_to_networkx import Neo4jToNetworkx

load_dotenv()

neo4jData = GraphDatabase.driver(
    os.getenv("NEO4J_DSN"),
    auth=(os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD")),
)

In [8]:
def get_all_session_ids():
    with neo4jData.session() as session:
        result = session.run("MATCH (n) RETURN DISTINCT n.session_id")
        return [record["n.session_id"] for record in result]

In [9]:
def get_image_ids_for_session(session_id):
    with neo4jData.session() as session:
        result = session.run("MATCH (n) WHERE n.session_id = $session_id RETURN DISTINCT n.image_id", session_id=session_id)
        return [record["n.image_id"] for record in result if record["n.image_id"] is not None]

In [10]:
session_to_image_ids: Dict[str, List[str]] = {}
for session_id in get_all_session_ids():
    if session_id is not None:  # Skip None values
        session_to_image_ids[session_id] = get_image_ids_for_session(session_id)

session_to_image_graphs = {}
with neo4jData.session() as session:
    for session_id, image_ids in session_to_image_ids.items():
        session_graphs = []
        for image_id in image_ids:
            if image_id is not None:  # Skip None values
                try:
                    graph = Neo4jToNetworkx.extract_image_graph(session, image_id)
                    session_graphs.append(graph)
                except Exception as e:
                    print(f"Error extracting graph for image {image_id} in session {session_id}: {e}")
        session_to_image_graphs[session_id] = session_graphs

In [19]:
def save_session_to_image_graphs(session_to_image_graphs):
    base_folder = "data"
    if not os.path.exists(base_folder):
        os.makedirs(base_folder)
    
    # Group graphs by session_id
    session_graphs = {}
    for session_id, graphs in session_to_image_graphs.items():
        if session_id not in session_graphs:
            session_graphs[session_id] = []
        for graph in graphs:
            session_graphs[session_id].append(nx.node_link_data(graph))
    
    # Save all graphs for each session in a single file
    for session_id, graphs in session_graphs.items():
        filename = os.path.join(base_folder, f"{session_id}.json")
        
        # Save all graphs for this session as a list in one JSON file
        with open(filename, 'w') as f:
            json.dump(graphs, f, indent=2)
        
        print(f"Saved {len(graphs)} graphs for session {session_id} to {filename}")

In [20]:
save_session_to_image_graphs(session_to_image_graphs)

/Users/mlapin/anaconda3/envs/natural_agi/lib/python3.11/site-packages/networkx/readwrite/json_graph/node_link.py:142: FutureWarning: 
The default value will be `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_data(G, edges="links") to preserve current behavior, or
  nx.node_link_data(G, edges="edges") for forward compatibility.
  warnings.warn(


Saved 2 graphs for session 5_3 to data/5_3.json
Saved 2 graphs for session 2_2 to data/2_2.json
Saved 2 graphs for session 6_2 to data/6_2.json
Saved 2 graphs for session 6_3 to data/6_3.json
Saved 0 graphs for session 7_1 to data/7_1.json
Saved 2 graphs for session 7_2 to data/7_2.json
Saved 2 graphs for session 2_4 to data/2_4.json
Saved 2 graphs for session 3_1 to data/3_1.json
Saved 0 graphs for session 8_1 to data/8_1.json
Saved 2 graphs for session 8_2 to data/8_2.json
Saved 2 graphs for session 3_3 to data/3_3.json
Saved 2 graphs for session 8_3 to data/8_3.json
Saved 2 graphs for session 9_1 to data/9_1.json
Saved 2 graphs for session 9_2 to data/9_2.json
Saved 2 graphs for session 9_3 to data/9_3.json
Saved 1 graphs for session test to data/test.json
Saved 2 graphs for session 3_2 to data/3_2.json
Saved 2 graphs for session 4_1 to data/4_1.json
Saved 2 graphs for session 4_2 to data/4_2.json
Saved 2 graphs for session 5_1 to data/5_1.json
Saved 2 graphs for session 5_2 to data